# 02 - Train and evaluate a Sentinel-1 land cover model

A U-Net (ResNet-34 encoder, trained from scratch) reads two radar bands (VV, VH) and labels every pixel with one of 7 classes.

Two variants differ in one thing only, set by `MONTHLY_ONLY` below:
- **A, monthly only** (`MONTHLY_ONLY = True`): pixels whose label was filled from the yearly map are ignored in training.
- **B, hybrid** (`MONTHLY_ONLY = False`): filled pixels are used, with a smaller loss weight.

Run the notebook once per variant. Both are scored on the same pixels, real monthly labels only, so the comparison is fair.

## 1. Settings

In [ ]:
MONTHLY_ONLY = False   # True = variant A (monthly labels only), False = variant B (hybrid labels)
FILLED_WEIGHT = 0.3    # loss weight of pixels whose label was filled from the yearly map (hybrid run only)
EPOCHS = 15
BATCH = 16
LR = 1e-3
CONTEXT = False        # True = variant C: also give the model the site's yearly median radar (4 input channels)
AGREE = False          # True = keep a yearly (filled) label only where ESA WorldCover agrees with it (label-quality filter)
FILTER = None          # None, "smooth" (light speckle filter) or "sharpen" (unsharp mask) applied to the radar before scaling
YEARS = None           # None = every year found for each site; (2024,) reproduces the earlier single-year runs
QUANTIZE = True        # store radar as 8-bit numbers: half the memory, which the bigger dataset needs
SEED = 0               # change it to repeat a run; SEED = 0 keeps the plain file names
SMOKE = False          # True = tiny quick run just to check that everything works
NAME = ("monthly_only" if MONTHLY_ONLY else "hybrid") + ("_ctx" if CONTEXT else "") + ("_agree" if AGREE else "") + ("_" + FILTER if FILTER else "") + (f"_e{EPOCHS}" if EPOCHS != 15 else "") + ("_y" + "".join(str(y)[2:] for y in YEARS) if YEARS else "") + (f"_seed{SEED}" if SEED else "") + ("_smoke" if SMOKE else "")

## 2. Data

`LULCDataset` (in `lulc.py`) cuts 256x256 windows and gives, for each one: scaled radar `x`, the 7-class label `y`
(255 = ignore), the label's confidence, and `filled` (True where the label came from the yearly map).

In [ ]:
import json
import os
import queue
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

from lulc import CLASSES, IGNORE, LULCDataset

C = len(CLASSES)
cuda = torch.cuda.is_available()
device = "cuda" if cuda else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", device, torch.cuda.get_device_name(0) if cuda else "")

torch.manual_seed(SEED)
np.random.seed(SEED)
train_ds, val_ds, test_ds = (LULCDataset(r, context=CONTEXT, agree=AGREE, years=YEARS, quantize=QUANTIZE, filter=FILTER) for r in ("train", "val", "test"))
if SMOKE:
    EPOCHS = 1
    for ds in (train_ds, val_ds, test_ds):
        ds.items = ds.items[::max(1, len(ds.items) // 64)][:64]   # spread over all sites and months
print({"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)}, "patches")


PREFETCH = cuda   # data in RAM: build the next batches in a background thread. On a slow GPU it made no difference
                  # (a step is mostly GPU time); on a fast GPU it may help. Set to False to compare epoch times.


def prefetch(gen, depth=3):
    """Runs a batch generator in a background thread, so the next batches are already built while the GPU works on
    the current one. An error in the thread is handed to the caller instead of hanging."""
    q = queue.Queue(maxsize=depth)
    end = object()

    def work():
        try:
            for item in gen:
                q.put(item)
            q.put(end)
        except BaseException as e:
            q.put(e)

    threading.Thread(target=work, daemon=True).start()
    while True:
        item = q.get()
        if item is end:
            return
        if isinstance(item, BaseException):
            raise item
        yield item


class GPUData:
    """A whole split held in memory and cut into batches by indexing, with no per-sample work. It lives on the GPU
    when it fits with room to spare (the GPU is shared and other people use it), otherwise on the CPU, and each batch
    is sent to the GPU. Either way a batch is a single indexing operation, not a Python loop."""

    def __init__(self, ds):
        self.p = ds.patch
        self.div = 255.0 if ds.quantize else 1.0                      # 8-bit radar is read back as 0..1
        radar = torch.from_numpy(np.stack([a.transpose(1, 2, 0) for a in ds.radar]))   # (N, H, W, channels)
        label = torch.from_numpy(np.stack(ds.label))                                    # (N, H, W) uint8
        filled = torch.from_numpy(np.stack(ds.filled))                                  # (N, H, W) bool
        need = sum(t.numel() * t.element_size() for t in (radar, label, filled))
        free = torch.cuda.mem_get_info()[0] if cuda else 0
        self.where = device if need < 0.4 * free else "cpu"
        self.radar, self.label, self.filled = (t.to(self.where) for t in (radar, label, filled))
        self.items = torch.tensor(ds.items, dtype=torch.long, device=self.where)        # (M, 3): site-month, row, col
        self.month = torch.tensor([ds.month[k] for k, _, _ in ds.items], device=self.where) - 1   # (M,) 0..11
        ds.radar = ds.conf = None                                                       # free the copies in the dataset object
        print(f"{len(self):,} windows, {need / 1e9:.1f} GB of radar and labels kept on {self.where}")

    def __len__(self):
        return len(self.items)

    def _cut(self, size, shuffle):
        """Cuts the batches by indexing, on the device where the data lives. No transfer to the GPU here."""
        order = torch.randperm(len(self), device=self.where) if shuffle else torch.arange(len(self), device=self.where)
        stop = len(order) - size + 1 if shuffle else len(order)       # drop the last short batch only when training
        ar = torch.arange(self.p, device=self.where)
        for i in range(0, stop, size):
            idx = order[i:i + size]
            k, r, c = self.items[idx].unbind(1)
            kk, rows, cols = k[:, None, None], (r[:, None] + ar)[:, :, None], (c[:, None] + ar)[:, None, :]
            yield (self.radar[kk, rows, cols].permute(0, 3, 1, 2).contiguous(),     # (B, channels, p, p)
                   self.label[kk, rows, cols], self.filled[kk, rows, cols], self.month[idx])

    def batches(self, size, shuffle):
        cut = self._cut(size, shuffle)
        if self.where == "cpu" and PREFETCH:
            cut = prefetch(cut)                                       # build the next batches while the GPU is busy
        for x, y, filled, month in cut:
            yield x.to(device).float() / self.div, y.long().to(device), filled.to(device), month.to(device)


train_data, val_data, test_data = GPUData(train_ds), GPUData(val_ds), GPUData(test_ds)
steps_per_epoch = len(train_data) // BATCH

### A picture of the data

**Left:** how many months of radar each training site has, split by year. **Right:** what share of the labelled pixels belongs to each class, for train, validation and test (all years, after the yearly fill). If the test bars look very different from the train bars, the model is being tested on a different kind of place than it learned from.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
sites = sorted(set(train_ds.site))
bottom = np.zeros(len(sites))
for y in sorted(set(train_ds.year)):
    counts = np.array([sum(1 for s_, y_ in zip(train_ds.site, train_ds.year) if s_ == site and y_ == y) for site in sites])
    ax[0].bar(sites, counts, bottom=bottom, label=str(y))
    bottom += counts
ax[0].set_ylabel("months of radar"); ax[0].set_title("training data per site and year"); ax[0].legend(title="year")
ax[0].tick_params(axis="x", rotation=75, labelsize=8)

width = 0.27
for i, (name, ds_) in enumerate((("train", train_ds), ("val", val_ds), ("test", test_ds))):
    counts = np.sum([np.bincount(l.ravel(), minlength=256)[:C] for l in ds_.label], axis=0)
    ax[1].bar(np.arange(C) + (i - 1) * width, counts / counts.sum(), width, label=name)
ax[1].set_xticks(range(C)); ax[1].set_xticklabels(CLASSES, rotation=30, ha="right")
ax[1].set_ylabel("share of labelled pixels"); ax[1].set_title("class shares per split"); ax[1].legend()
plt.tight_layout()
plt.show()

### What the radar filters do

The same piece of a radar image with no filter, with `smooth` (a light speckle filter) and with `sharpen` (an unsharp mask). Sharpening boosts the edges, but it boosts the speckle grain just as much. Set `FILTER` at the top to train with one of them.

In [ ]:
import glob

from lulc import filter_radar

site = sorted(set(val_ds.site))[0]
raw = np.load(sorted(glob.glob(f"data/{site}_2024-*.npz"))[5])["radar"][:1]     # VV in dB, one mid-year month
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, kind in zip(ax, (None, "smooth", "sharpen")):
    a.imshow(filter_radar(raw, kind)[0, 300:556, 300:556], cmap="gray", vmin=-25, vmax=5)
    a.set_title(f"{site}: {kind or 'no filter'} (VV, 256x256 piece)"); a.axis("off")
plt.tight_layout()
plt.show()

## 3. Class weights

Rare classes (bare, water) would be drowned out by built and trees. Each class is weighted by the inverse square root of
how common it is in the training data, which is a milder correction than full inverse frequency.

In [ ]:
lab = np.concatenate([l.ravel() for l in train_ds.label])
freq = np.bincount(lab, minlength=256)[:C] / (lab != IGNORE).sum()
class_w = 1 / np.sqrt(np.maximum(freq, 1e-6))
class_w = torch.tensor(class_w / class_w.mean(), dtype=torch.float32).to(device)
print({c: round(float(w), 2) for c, w in zip(CLASSES, class_w.cpu())})

## 4. Model, loss, optimiser

- **Model:** `smp.Unet` with a ResNet-34 encoder and no pretrained weights. 2 input channels (4 with `CONTEXT`), 7 output classes.
- **Loss:** per-pixel cross-entropy with class weights. Ignored pixels contribute nothing. In the hybrid run, filled pixels are multiplied by `FILLED_WEIGHT`.
- **Learning rate:** starts at `LR` and shrinks smoothly to zero over training (cosine schedule).
- **Confidence** is not used in this first iteration.

In [ ]:
model = smp.Unet("resnet34", encoder_weights=None, in_channels=train_ds.channels, classes=C).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * steps_per_epoch)
ce = nn.CrossEntropyLoss(weight=class_w, ignore_index=IGNORE, reduction="none")


def batch_loss(logits, y, filled):
    """Weighted cross-entropy averaged over the pixels that have a usable label."""
    if MONTHLY_ONLY:
        y = y.masked_fill(filled, IGNORE)            # variant A: filled pixels give no feedback
        w = torch.ones(y.shape, device=y.device)
    else:
        w = 1.0 - (1.0 - FILLED_WEIGHT) * filled.float()   # variant B: filled pixels count for less
    valid = (y != IGNORE).float() * w
    return (ce(logits, y) * w).sum() / valid.sum().clamp(min=1)


def augment(x, y, filled):
    """Random 90-degree rotation and flip, the same for the whole batch (radar has no 'up')."""
    k = np.random.randint(4)
    x, y, filled = (torch.rot90(t, k, (-2, -1)) for t in (x, y, filled))
    if np.random.rand() < 0.5:
        x, y, filled = (torch.flip(t, (-1,)) for t in (x, y, filled))
    return x, y, filled

## 5. Scoring

Everything is scored on **real monthly labels only** (pixels with a label that is not filled), whichever variant was trained.
A confusion matrix counts, for every true class, which class the model predicted. All the scores come from it:
overall accuracy (OA), IoU and F1 per class, and mean IoU (mIoU) over the classes that appear.

In [ ]:
@torch.no_grad()
def confusion(data):
    """Returns the confusion matrix over the whole set, and one per month (1..12).
    Every counted pixel gets one number, month * C * C + true * C + predicted, and one bincount tallies them all."""
    model.eval()
    counts = torch.zeros(12 * C * C, dtype=torch.long, device=device)
    for x, y, filled, month in data.batches(BATCH * 2, shuffle=False):
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=cuda):
            pred = model(x).argmax(1)
        keep = (y != IGNORE) & ~filled
        key = month[:, None, None] * C * C + y * C + pred
        counts += torch.bincount(key[keep], minlength=counts.numel())
    cms = counts.view(12, C, C).cpu().numpy()
    return cms.sum(0), {m: cms[m - 1] for m in range(1, 13)}


def scores(cm):
    tp = np.diag(cm).astype(float)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    iou = tp / np.maximum(tp + fp + fn, 1)
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    present = cm.sum(1) > 0
    return {"oa": tp.sum() / max(cm.sum(), 1), "miou": iou[present].mean() if present.any() else 0.0,
            "iou": iou, "f1": f1, "pixels": int(cm.sum())}


# self-check of the scoring: a perfect prediction scores 1, and a fully wrong one scores 0
_cm = np.diag(np.arange(1, C + 1) * 10)
assert scores(_cm)["oa"] == 1 and abs(scores(_cm)["miou"] - 1) < 1e-9
_cm = np.roll(_cm, 1, axis=1)
assert scores(_cm)["oa"] == 0


def plot_confusion(cm, name=""):
    cm = cm.astype(float)
    by_true = cm / np.maximum(cm.sum(1, keepdims=True), 1)    # each row sums to 1
    by_pred = cm / np.maximum(cm.sum(0, keepdims=True), 1)    # each column sums to 1
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    for a, m, title in ((ax[0], by_true, "of each TRUE class: share predicted as ..."),
                        (ax[1], by_pred, "of each PREDICTED class: share that truly is ...")):
        a.imshow(m, cmap="Blues", vmin=0, vmax=1)
        a.set_xticks(range(C)); a.set_xticklabels(CLASSES, rotation=45, ha="right")
        a.set_yticks(range(C)); a.set_yticklabels(CLASSES)
        a.set_xlabel("predicted class"); a.set_ylabel("true class")
        a.set_title(f"{name} {title}".strip())
        for i in range(C):
            for j in range(C):
                a.text(j, i, f"{m[i, j]:.2f}", ha="center", va="center", color="white" if m[i, j] > 0.5 else "black")
    plt.tight_layout()
    plt.show()

## 6. Train

In [ ]:
os.makedirs("weights", exist_ok=True)
history = {"loss": [], "val_oa": [], "val_miou": [], "lr": [], "seconds": []}
best = -1.0

for epoch in range(EPOCHS):
    t_epoch = time.time()
    model.train()
    running = []
    for x, y, filled, _ in train_data.batches(BATCH, shuffle=True):
        x, y, filled = augment(x, y, filled)
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=cuda):
            logits = model(x)
        loss = batch_loss(logits.float(), y, filled)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        running.append(loss.item())
    s = scores(confusion(val_data)[0])
    history["loss"].append(float(np.mean(running)))
    history["val_oa"].append(float(s["oa"]))
    history["val_miou"].append(float(s["miou"]))
    history["lr"].append(opt.param_groups[0]["lr"])
    history["seconds"].append(time.time() - t_epoch)
    if s["miou"] > best:                      # keep the weights that do best on validation sites
        best = s["miou"]
        torch.save(model.state_dict(), f"weights/{NAME}.pt")
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  loss {history['loss'][-1]:.3f}  val OA {s['oa']:.3f}  val mIoU {s['miou']:.3f}  ({history['seconds'][-1]:.0f}s)")

## 7. Training curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(history["loss"]); ax[0].set_title("training loss")
ax[1].plot(history["val_miou"], label="mIoU"); ax[1].plot(history["val_oa"], label="OA"); ax[1].legend(); ax[1].set_title("validation")
ax[2].plot(history["lr"]); ax[2].set_title("learning rate")
for a in ax: a.set_xlabel("epoch")
plt.show()

## 8. Test scores (test sites, real monthly labels only)

The test sites have not been used for training or for choosing weights. Scores are by class and by month. Months with few labelled pixels (the rainy season) rest on thin data, so the number of pixels is printed too.

In [ ]:
model.load_state_dict(torch.load(f"weights/{NAME}.pt", map_location=device))
total, per_month = confusion(test_data)
t = scores(total)
vtotal, _ = confusion(val_data)            # validation numbers decide whether a change worked; test is the final exam
v = scores(vtotal)
water = CLASSES.index("water")
v_water_precision = vtotal[water, water] / max(vtotal[:, water].sum(), 1)   # share of predicted water that truly is water
print(f"[{NAME}] VALIDATION mIoU {v['miou']:.3f}  water IoU {v['iou'][water]:.3f}  water precision {v_water_precision:.3f}\n")
print(f"[{NAME}] test OA {t['oa']:.3f}  mIoU {t['miou']:.3f}  over {t['pixels']:,} labelled pixels\n")
print(f"{'class':20s}{'IoU':>7s}{'F1':>7s}{'pixels':>12s}")
for i, c in enumerate(CLASSES):
    print(f"{c:20s}{t['iou'][i]:7.3f}{t['f1'][i]:7.3f}{int(total[i].sum()):12,d}")
print(f"\n{'month':>5s}{'OA':>7s}{'mIoU':>7s}{'pixels':>12s}")
months = {}
for m, cm in per_month.items():
    s = scores(cm)
    months[m] = {"oa": float(s["oa"]), "miou": float(s["miou"]), "pixels": s["pixels"]}
    print(f"{m:5d}{s['oa']:7.3f}{s['miou']:7.3f}{s['pixels']:12,d}")
json.dump({"name": NAME, "oa": float(t["oa"]), "miou": float(t["miou"]), "iou": t["iou"].tolist(), "f1": t["f1"].tolist(),
           "val_miou": float(v["miou"]), "val_water_iou": float(v["iou"][water]), "val_water_precision": float(v_water_precision),
           "months": months, "history": history}, open(f"results_{NAME}.json", "w"))

### Confusion matrix on the test sites

Rows are the true classes, columns the predicted ones. Left: of each true class, what share was predicted as each class
(shows what the model misses). Right: of each predicted class, what share truly belongs to each class (shows false alarms).

In [ ]:
plot_confusion(total, "test:")

## 9. Look at the mistakes

Tune and diagnose on the **validation** sites, not the test sites, so the test score stays an honest final exam.
Set `SPLIT`, `SITE` and `MONTH` below. The plot shows the radar, the true label (real monthly labels only, white where there is none),
the prediction, and the pixels the model called water although they are not water.

In [ ]:
@torch.no_grad()
def predict_full(radar):
    """radar: (2, H, W) scaled, numpy. The U-Net needs sizes divisible by 32, so pad to the next multiple and crop back."""
    h, w = radar.shape[1:]
    x = torch.from_numpy(radar.astype(np.float32))[None].to(device)
    x = torch.nn.functional.pad(x, (0, -w % 32, 0, -h % 32), mode="reflect")
    model.eval()
    with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=cuda):
        pred = model(x).argmax(1)[0]
    return pred[:h, :w].cpu().numpy()


model.load_state_dict(torch.load(f"weights/{NAME}.pt", map_location=device))   # the best weights from training
print("validation confusion matrix")
plot_confusion(confusion(val_data)[0], "val:")

SPLIT, SITE, MONTH = "val", "warangal", 1       # change these to look elsewhere ("train", "val" or "test")
ds = {"train": train_ds, "val": val_ds, "test": test_ds}[SPLIT]
k = next(i for i in range(len(ds.site)) if ds.site[i] == SITE and ds.month[i] == MONTH)
data = {"train": train_data, "val": val_data, "test": test_data}[SPLIT]
radar = (data.radar[k].permute(2, 0, 1).float() / data.div).cpu().numpy()   # (channels, H, W), scaled 0..1
label, filled = ds.label[k], ds.filled[k]
pred = predict_full(radar)
real = (label != IGNORE) & ~filled                 # pixels with a real monthly label
water = CLASSES.index("water")
false_water = (pred == water) & real & (label != water)

cmap = plt.get_cmap("tab10", 10)
show = lambda a: np.ma.masked_equal(a, IGNORE)
fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
ax[0].imshow(radar[0], cmap="gray"); ax[0].set_title(f"{SITE} month {MONTH}: VV radar (scaled)")
ax[1].imshow(show(np.where(real, label, IGNORE)), cmap=cmap, vmin=0, vmax=9); ax[1].set_title("true label (real monthly labels)")
ax[2].imshow(pred, cmap=cmap, vmin=0, vmax=9); ax[2].set_title("model prediction")
ax[3].imshow(radar[0], cmap="gray", alpha=0.5); ax[3].imshow(np.ma.masked_equal(false_water.astype(int), 0), cmap="autumn", vmin=0, vmax=1)
ax[3].set_title("false water (predicted water, truly something else)")
for a in ax: a.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(i)) for i in range(C)]
fig.legend(handles, CLASSES, loc="lower center", ncol=C)
plt.show()

n = max(int(real.sum()), 1)
print(f"real-label pixels: {real.sum():,}  false-water pixels: {false_water.sum():,} ({false_water.sum() / n:.1%} of them)")
if false_water.any():
    share = np.bincount(label[false_water], minlength=C) / false_water.sum()
    print("what the false water truly is:", {c: round(float(s), 2) for c, s in zip(CLASSES, share) if s > 0.01})

### Where the model is weak: a score for every part of each site

Each site is cut into `GRID` x `GRID` cells. Colour = the share of real-label pixels the model got right in that cell, over all months (green = good, red = poor). Grey = too few labelled pixels to say. Validation sites first, then test sites.

In [ ]:
GRID = 4


def grid_scores(ds, data):
    """Per site: [pixels right, pixels counted] in each grid cell, over all months with real labels."""
    per = {}
    for k in range(len(ds.site)):
        keep = (ds.label[k] != IGNORE) & ~ds.filled[k]
        if not keep.any():
            continue
        pred = predict_full((data.radar[k].permute(2, 0, 1).float() / data.div).cpu().numpy())
        h, w = keep.shape
        cell = lambda a: a[:h // GRID * GRID, :w // GRID * GRID].astype(np.int64).reshape(GRID, h // GRID, GRID, w // GRID).sum(axis=(1, 3))
        right, total = per.setdefault(ds.site[k], [np.zeros((GRID, GRID)), np.zeros((GRID, GRID))])
        right += cell((pred == ds.label[k]) & keep)
        total += cell(keep)
    return per


for split_name, ds_, data_ in (("validation", val_ds, val_data), ("test", test_ds, test_data)):
    per = grid_scores(ds_, data_)
    cols = min(5, len(per)); rows = -(-len(per) // cols)          # wrap into rows of at most 5 sites
    fig, ax = plt.subplots(rows, cols, figsize=(4.2 * cols, 4.2 * rows))
    ax = np.atleast_1d(ax).ravel()
    for a in ax[len(per):]:
        a.axis("off")
    for a, (site_, (right, total)) in zip(ax, per.items()):
        acc = np.where(total >= 1000, right / np.maximum(total, 1), np.nan)
        im = a.imshow(np.ma.masked_invalid(acc), cmap="RdYlGn", vmin=0, vmax=1)
        a.set_facecolor("lightgrey")
        for i in range(GRID):
            for j in range(GRID):
                if not np.isnan(acc[i, j]):
                    a.text(j, i, f"{acc[i, j]:.0%}", ha="center", va="center", fontsize=9)
        a.set_title(f"{site_}: {right.sum() / max(total.sum(), 1):.0%} overall"); a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"share of pixels right per grid cell, {split_name} sites")
    plt.tight_layout()
    plt.show()

## 10. Compare the two variants

Lists every saved run side by side (nan = that run was saved before the validation numbers were recorded). The validation block is the one to judge changes by; the test block is read once at the end.

In [ ]:
names = sorted(f[8:-5] for f in os.listdir(".") if f.startswith("results_") and f.endswith(".json"))
res = {n: json.load(open(f"results_{n}.json")) for n in names}
if not res:
    print("no results_*.json found; train and score a variant first")
else:
    cols = "".join(f"{n:>18s}" for n in names)
    print(f"{'VALIDATION (decide with these)':30s}{cols}")
    print(f"{'  mIoU':30s}" + "".join(f"{r.get('val_miou', float('nan')):18.3f}" for r in res.values()))
    print(f"{'  water IoU':30s}" + "".join(f"{r.get('val_water_iou', float('nan')):18.3f}" for r in res.values()))
    print(f"{'  water precision':30s}" + "".join(f"{r.get('val_water_precision', float('nan')):18.3f}" for r in res.values()))
    print(f"\n{'TEST (final exam)':30s}{cols}")
    print(f"{'  OA':30s}" + "".join(f"{r['oa']:18.3f}" for r in res.values()))
    print(f"{'  mIoU':30s}" + "".join(f"{r['miou']:18.3f}" for r in res.values()))
    for i, c in enumerate(CLASSES):
        print(f"{'  IoU ' + c:30s}" + "".join(f"{r['iou'][i]:18.3f}" for r in res.values()))
    print(f"\n{'test mIoU by month':30s}{cols}")
    for m in range(1, 13):
        print(f"{'  month ' + str(m):30s}" + "".join(f"{r['months'][str(m)]['miou']:18.3f}" for r in res.values()))
    # the same numbers as pictures. Left: validation sites. Right: test sites, one group of bars per class.
    x = np.arange(len(names))
    fig, ax = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1, 2]})
    ax[0].bar(x - 0.2, [r.get("val_miou", np.nan) for r in res.values()], 0.4, label="mIoU")
    ax[0].bar(x + 0.2, [r.get("val_water_iou", np.nan) for r in res.values()], 0.4, label="water IoU")
    ax[0].set_xticks(x); ax[0].set_xticklabels(names, rotation=45, ha="right")
    ax[0].set_title("validation sites (higher is better)"); ax[0].legend()
    w = 0.8 / len(names)
    for i, (n, r) in enumerate(res.items()):
        ax[1].bar(np.arange(C) + (i - (len(names) - 1) / 2) * w, r["iou"], w, label=n)
    ax[1].set_xticks(range(C)); ax[1].set_xticklabels(CLASSES, rotation=25, ha="right")
    ax[1].set_ylabel("IoU"); ax[1].set_title("test sites: IoU per class (higher is better)"); ax[1].legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()